<a href="https://colab.research.google.com/github/codewithabir/C-plus-plus/blob/main/Lunar_lander.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!nvidia-smi
!apt-get update -qq
!apt-get install -y swig cmake
!pip install -r https://raw.githubusercontent.com/huggingface/deep-rl-class/main/notebooks/unit1/requirements-unit1.txt

Fri Nov  6 17:16:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -U "stable-baselines3[extra]" "gymnasium[box2d]" huggingface_hub
import gymnasium as gym
import stable_baselines3 as sb3
import torch

print("Gymnasium:", gym.__version__)
print("Stable Baselines3:", sb3.__version__)
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

env = gym.make("LunarLander-v3")

observation, info = env.reset()

print("Observation:", observation)
print("Observation Shape:", observation.shape)
print("Action Space:", env.action_space)

env.close()

In [ ]:
from pyvirtualdisplay import Display

virtual_display = Display(
    visible=0,
    size=(1400, 900)
)

virtual_display.start()

print("Virtual Display Started Successfully!")

In [ ]:
import gymnasium as gym

env = gym.make("LunarLander-v3", render_mode="rgb_array")

observation, info = env.reset()

print("Observation:", observation)
print("Observation Shape:", observation.shape)
print("Action Space:", env.action_space)

env.close()

In [ ]:
from stable_baselines3.common.env_util import make_vec_env

env = make_vec_env(
    "LunarLander-v3",
    n_envs=16
)

print("Number of Environments:", env.num_envs)
print("Observation Space:", env.observation_space)
print("Action Space:", env.action_space)

In [ ]:
from stable_baselines3 import PPO

model = PPO(
    policy="MlpPolicy",
    env=env,
    n_steps=1024,
    batch_size=64,
    n_epochs=4,
    gamma=0.999,
    gae_lambda=0.98,
    ent_coef=0.01,
    verbose=1,
    device="cpu"
)

print("PPO Model Created Successfully!")

In [ ]:
model.learn(total_timesteps=1_000_000)

model.save("ppo-LunarLander-v3")

print("Training Completed!")

In [ ]:
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
import gymnasium as gym

eval_env = Monitor(gym.make("LunarLander-v3"))

mean_reward, std_reward = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=10,
    deterministic=True
)

print("Mean Reward:", mean_reward)
print("Standard Deviation:", std_reward)

eval_env.close()

In [17]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from IPython.display import Video, display
import glob

video_env = gym.make(
    "LunarLander-v3",
    render_mode="rgb_array"
)

video_env = RecordVideo(
    video_env,
    video_folder="./lander_video",
    episode_trigger=lambda episode_id: True
)

obs, info = video_env.reset()

terminated = False
truncated = False

while not (terminated or truncated):
    action, _ = model.predict(
        obs,
        deterministic=True
    )

    obs, reward, terminated, truncated, info = video_env.step(action)

video_env.close()

video_files = glob.glob("./lander_video/*.mp4")

display(Video(video_files[0], embed=True))

/usr/local/lib/python3.13/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [18]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [19]:
import shutil
import os

source = "/content/ppo-LunarLander-v3.zip"

destination = "/content/drive/MyDrive/LunarLander_Project"

os.makedirs(destination, exist_ok=True)

shutil.copy2(
    source,
    destination + "/ppo-LunarLander-v3.zip"
)

print("Model Successfully Saved to Google Drive!")

Model Successfully Saved to Google Drive!


In [20]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from IPython.display import Video, display
import os

video_env = gym.make(
    "LunarLander-v3",
    render_mode="rgb_array"
)

video_env = RecordVideo(
    video_env,
    video_folder="/content/lander_video",
    episode_trigger=lambda episode_id: True
)

obs, info = video_env.reset()

terminated = False
truncated = False
total_reward = 0

while not (terminated or truncated):

    action, _ = model.predict(
        obs,
        deterministic=True
    )

    obs, reward, terminated, truncated, info = video_env.step(action)

    total_reward += reward

video_env.close()

print("Episode Reward:", total_reward)
print("Video Generated Successfully!")

display(Video(
    "/content/lander_video/rl-video-episode-0.mp4",
    embed=True
))

/usr/local/lib/python3.13/dist-packages/gymnasium/wrappers/rendering.py:298: UserWarning: WARN: Overwriting existing videos at /content/lander_video folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Episode Reward: 227.40818804344065
Video Generated Successfully!


In [21]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from IPython.display import Video, display
import os

video_env = gym.make(
    "LunarLander-v3",
    render_mode="rgb_array"
)

video_env = RecordVideo(
    video_env,
    video_folder="/content/lander_video",
    episode_trigger=lambda episode_id: True
)

obs, info = video_env.reset()

terminated = False
truncated = False
total_reward = 0

while not (terminated or truncated):

    action, _ = model.predict(
        obs,
        deterministic=True
    )

    obs, reward, terminated, truncated, info = video_env.step(action)

    total_reward += reward

video_env.close()

print("Episode Reward:", total_reward)
print("Video Generated Successfully!")

display(Video(
    "/content/lander_video/rl-video-episode-0.mp4",
    embed=True
))

Episode Reward: 294.7261182870101
Video Generated Successfully!


In [22]:
shutil.copy2(
    "/content/lander_video/rl-video-episode-0.mp4",
    "/content/drive/MyDrive/LunarLander_Project/lander-demo.mp4"
)

print("Video Saved to Google Drive!")

Video Saved to Google Drive!


In [23]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from IPython.display import Video, display
import os

video_env = gym.make(
    "LunarLander-v3",
    render_mode="rgb_array"
)

video_env = RecordVideo(
    video_env,
    video_folder="/content/lander_video",
    episode_trigger=lambda episode_id: True
)

obs, info = video_env.reset()

terminated = False
truncated = False
total_reward = 0

while not (terminated or truncated):

    action, _ = model.predict(
        obs,
        deterministic=True
    )

    obs, reward, terminated, truncated, info = video_env.step(action)

    total_reward += reward

video_env.close()

print("Episode Reward:", total_reward)
print("Video Generated Successfully!")

display(Video(
    "/content/lander_video/rl-video-episode-0.mp4",
    embed=True
))

Episode Reward: 251.55775131426014
Video Generated Successfully!


In [24]:
shutil.copy2(
    "/content/lander_video/rl-video-episode-0.mp4",
    "/content/drive/MyDrive/LunarLander_Project/lander-demo.mp4"
)

print("Video Saved to Google Drive!")

Video Saved to Google Drive!
